# 1.Imports


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
import seaborn as sns

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pmdarima import auto_arima
from pmdarima.model_selection import train_test_split as ts_train_test_split

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.size': 11
})


# 2. Load and Preprocessing


In [ ]:
df = pd.read_csv('powerconsumption.csv', parse_dates=['Datetime'])
df.set_index('Datetime', inplace=True)
df.index.freq = None  

print(f'Shape: {df.shape}')
print(f'Período: {df.index.min()} → {df.index.max()}')
print(f'\nColumnas:\n{df.dtypes}')
df.head()

In [ ]:
df_hourly = df.resample('H').mean()

print(f'Registros originales (10 min): {len(df):,}')
print(f'Registros horarios:            {len(df_hourly):,}')
print(f'Período: {df_hourly.index.min()} → {df_hourly.index.max()}')

series = df_hourly['PowerConsumption_Zone1']
print(f'\nSerie objetivo: PowerConsumption_Zone1')
print(f'Media: {series.mean():.1f} | Std: {series.std():.1f}')

# 3.Train/Test split


In [ ]:
TEST_HOURS = 168  

train = series.iloc[:-TEST_HOURS]
test  = series.iloc[-TEST_HOURS:]

print(f'Tamaño train : {len(train):,} horas  ({len(train)/24:.0f} días)')
print(f'Tamaño test  : {len(test):,} horas  ({len(test)/24:.0f} días)')
print(f'Train: {train.index.min()} → {train.index.max()}')
print(f'Test : {test.index.min()}  → {test.index.max()}')

# 4. AutoArima without seasonality


In [ ]:
model_no_seasonal = auto_arima(
    train,
    seasonal=False,         
    stepwise=True,            
    information_criterion='aic',
    error_action='ignore',
    suppress_warnings=True,
    trace=True               
)

print(f'\n✅ Mejor modelo (sin estacionalidad): ARIMA{model_no_seasonal.order}')
print(f'   AIC: {model_no_seasonal.aic():.2f}')
print(f'   BIC: {model_no_seasonal.bic():.2f}')

In [ ]:
print(model_no_seasonal.summary())

In [ ]:
model_no_seasonal.plot_diagnostics(figsize=(14, 8))
plt.suptitle('Diagnóstico de Residuos — AutoARIMA sin estacionalidad', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
forecast_no_seasonal, conf_int_no_seasonal = model_no_seasonal.predict(
    n_periods=TEST_HOURS,
    return_conf_int=True,
    alpha=0.05
)

forecast_no_seasonal = pd.Series(forecast_no_seasonal, index=test.index)

# Métricas
mae_ns  = mean_absolute_error(test, forecast_no_seasonal)
rmse_ns = np.sqrt(mean_squared_error(test, forecast_no_seasonal))
mape_ns = np.mean(np.abs((test.values - forecast_no_seasonal.values) / test.values)) * 100

print('📈 Resultados — AutoARIMA SIN Estacionalidad')
print(f'   Modelo : ARIMA{model_no_seasonal.order}')
print(f'   MAE    : {mae_ns:.2f} kW')
print(f'   RMSE   : {rmse_ns:.2f} kW')
print(f'   MAPE   : {mape_ns:.2f} %')

# 5. AutoArima with seasonality

In [ ]:
train_seasonal = train[-2160:]  # 90 días x 24 horas = 2160 observaciones hacemos con observaciones de 90 dias porque si utilizamos el año da memory error
#Copnsideramos que un periodo de 90 dias sigue siendo util para conseguir ver un patron o tendencia en el uso de energia diario

print('🔍 Ajustando AutoARIMA CON estacionalidad (m=24)...')
print(f'   Usando train reducido: {len(train_seasonal)} horas ({len(train_seasonal)//24} días)')
print('   (puede tardar varios minutos)\n')

model_seasonal = auto_arima(
    train_seasonal,
    seasonal=True,            # Con componente estacional
    m=24,                     # Ciclo diario: 24 horas
    stepwise=True,
    information_criterion='aic',
    error_action='ignore',
    suppress_warnings=True,
    trace=True
)

print(f'\n✅ Mejor modelo (con estacionalidad): SARIMA{model_seasonal.order}x{model_seasonal.seasonal_order}')
print(f'   AIC: {model_seasonal.aic():.2f}')
print(f'   BIC: {model_seasonal.bic():.2f}')

In [ ]:
print(model_seasonal.summary())

In [ ]:
model_seasonal.plot_diagnostics(figsize=(14, 8))
plt.suptitle('Diagnóstico de Residuos — AutoARIMA con estacionalidad', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
forecast_seasonal, conf_int_seasonal = model_seasonal.predict(
    n_periods=TEST_HOURS,
    return_conf_int=True,
    alpha=0.05
)

forecast_seasonal = pd.Series(forecast_seasonal, index=test.index)

# Métricas
mae_s  = mean_absolute_error(test, forecast_seasonal)
rmse_s = np.sqrt(mean_squared_error(test, forecast_seasonal))
mape_s = np.mean(np.abs((test.values - forecast_seasonal.values) / test.values)) * 100

print('📈 Resultados — AutoARIMA CON Estacionalidad')
print(f'   Modelo : SARIMA{model_seasonal.order}x{model_seasonal.seasonal_order}')
print(f'   MAE    : {mae_s:.2f} kW')
print(f'   RMSE   : {rmse_s:.2f} kW')
print(f'   MAPE   : {mape_s:.2f} %')

# 6. Visual Comparison

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)
fig.suptitle('AutoARIMA — Predicción vs Valores Reales (últimos 7 días)', fontsize=14, fontweight='bold')

for ax, (fc, ci, title, color) in zip(axes, [
    (forecast_no_seasonal, conf_int_no_seasonal,
     f'Sin Estacionalidad — ARIMA{model_no_seasonal.order}', '#E91E63'),
    (forecast_seasonal, conf_int_seasonal,
     f'Con Estacionalidad — SARIMA{model_seasonal.order}x{model_seasonal.seasonal_order}', '#4CAF50'),
]):
    ax.plot(test.index, test.values, color='#2196F3', linewidth=1.8, label='Real', zorder=3)
    ax.plot(test.index, fc.values,   color=color,    linewidth=1.5, linestyle='--', label='Predicción', zorder=2)
    ax.fill_between(
        test.index,
        ci[:, 0], ci[:, 1],
        color=color, alpha=0.15, label='IC 95%'
    )
    ax.set_title(title, fontsize=12)
    ax.set_ylabel('Consumo (kW)')
    ax.legend(loc='upper right')
    ax.xaxis.set_major_locator(mdates.DayLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))

plt.tight_layout()
plt.show()

In [ ]:

results = pd.DataFrame({
    'Modelo': [
        f'ARIMA{model_no_seasonal.order}\n(sin estacionalidad)',
        f'SARIMA{model_seasonal.order}x{model_seasonal.seasonal_order}\n(con estacionalidad m=24)'
    ],
    'AIC': [model_no_seasonal.aic(), model_seasonal.aic()],
    'BIC': [model_no_seasonal.bic(), model_seasonal.bic()],
    'MAE':  [mae_ns,  mae_s],
    'RMSE': [rmse_ns, rmse_s],
    'MAPE (%)': [mape_ns, mape_s]
})

print('=' * 70)
print('TABLA COMPARATIVA — AutoARIMA con y sin estacionalidad')
print('=' * 70)
print(results.to_string(index=False, float_format='{:.2f}'.format))
print('=' * 70)

# Resaltar mejor modelo
mejor_mae = 'CON estacionalidad' if mae_s < mae_ns else 'SIN estacionalidad'
mejor_rmse = 'CON estacionalidad' if rmse_s < rmse_ns else 'SIN estacionalidad'
print(f'\n🏆 Menor MAE  → {mejor_mae}')
print(f'🏆 Menor RMSE → {mejor_rmse}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Comparación de Métricas — AutoARIMA', fontsize=13, fontweight='bold')

labels = ['Sin\nestacionalidad', 'Con\nestacionalidad']
colors = ['#E91E63', '#4CAF50']

for ax, (metric_vals, metric_name) in zip(axes, [
    ([mae_ns, mae_s],   'MAE (kW)'),
    ([rmse_ns, rmse_s], 'RMSE (kW)'),
    ([mape_ns, mape_s], 'MAPE (%)'),
]):
    bars = ax.bar(labels, metric_vals, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
    ax.set_title(metric_name, fontsize=12)
    ax.set_ylim(0, max(metric_vals) * 1.3)
    for bar, val in zip(bars, metric_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(metric_vals)*0.02,
                f'{val:.2f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()